# Python — NumPy & Vectorization
---

<div style="padding:15px;border-left:8px solid #f093fb;background:#fdf0ff;border-radius:4px;">
<strong>Core Insight:</strong> NumPy operations run in C. A Python loop over 1M elements
takes ~seconds. NumPy does the same in milliseconds. This is the difference between a
pipeline that finishes in 2 minutes vs 2 hours — without changing the algorithm.
</div>

### Why It Matters for Data Engineering
Capacity analysis often involves large matrices of metrics × timestamps × servers.
At Citi: 1,000 timestamps × 6,000 servers = 6M values processed daily.
Python loop: 180 seconds. NumPy vectorized: 1.2 seconds.

## 🧠 Mental Models

| Model | The Story |
|-------|-----------|
| **The Factory Floor** | Python loop = one worker doing one task at a time. NumPy = assembly line of specialized C workers doing the same operation on all items simultaneously. |
| **The Recipe** | Broadcasting is like saying "add 1 tablespoon of salt to every bowl" — you don't enumerate each bowl, you describe the operation once. |
| **The Type Contract** | NumPy arrays have a fixed dtype. All elements are the same type, stored contiguously in memory. Python lists are pointers to objects scattered in heap. |

**The Rule:** Any time you write `for i in range(len(arr)):` on a NumPy array, ask:
*"Is there a vectorized equivalent?"* Usually yes.

In [ ]:
import numpy as np
import time

# ══════════════════════════════════════
# BENCHMARK: Python loop vs NumPy
# ══════════════════════════════════════

N = 1_000_000  # 1 million elements

# Python loop
data_list = list(range(N))
start = time.perf_counter()
result_loop = [x * 2.5 + 100 for x in data_list]
loop_time = time.perf_counter() - start

# NumPy vectorized
data_arr = np.arange(N, dtype=np.float64)
start = time.perf_counter()
result_np = data_arr * 2.5 + 100
np_time = time.perf_counter() - start

print(f"Python loop:     {loop_time*1000:.1f} ms")
print(f"NumPy vectorized:{np_time*1000:.1f} ms")
print(f"Speedup:         {loop_time/np_time:.0f}x")

# Memory layout
print(f"
NumPy array dtype: {data_arr.dtype}")
print(f"Memory (bytes):    {data_arr.nbytes:,}")
print(f"Contiguous:        {data_arr.flags['C_CONTIGUOUS']}")

## 📐 Broadcasting: The Key to Efficient Array Operations

Broadcasting lets NumPy operate on arrays with different shapes without copying data.

**The Rules:**
1. If arrays have different numbers of dimensions, prepend 1s to the smaller shape
2. Dimensions must either match or one of them must be 1
3. Output shape is the max of each dimension

```
Shape (6000, 1000)  — servers × timestamps (2D)
Shape (1000,)       — per-timestamp baseline (1D → broadcast to (1, 1000))
Result: (6000, 1000) — subtract each server's reading from the baseline
```

This is the **core operation** for anomaly detection: compare each server's
reading to the average across all servers at each timestamp.

In [ ]:
import numpy as np

# ══════════════════════════════════════
# CITI USE CASE: Capacity matrix analysis
# 6,000 servers × 1,000 timestamps = 6M values
# ══════════════════════════════════════

np.random.seed(42)
N_SERVERS = 6000
N_TIMESTAMPS = 1000

# Simulate CPU readings (percent utilization)
cpu_matrix = np.random.normal(loc=45, scale=15, size=(N_SERVERS, N_TIMESTAMPS))
cpu_matrix = np.clip(cpu_matrix, 0, 100)

# ── Vectorized Operations ──────────────────────────────────────────────────

# 1. Per-server statistics (axis=1 = across timestamps)
avg_cpu = cpu_matrix.mean(axis=1)          # shape: (6000,)
peak_cpu = cpu_matrix.max(axis=1)           # shape: (6000,)
p95_cpu = np.percentile(cpu_matrix, 95, axis=1)  # shape: (6000,)

# 2. Servers breaching 80% threshold (vectorized boolean mask)
THRESHOLD = 80.0
breaching_mask = avg_cpu > THRESHOLD        # shape: (6000,), dtype: bool
n_breaching = breaching_mask.sum()
breach_indices = np.where(breaching_mask)[0]

print(f"Servers analyzed:   {N_SERVERS:,}")
print(f"Timestamps:         {N_TIMESTAMPS:,}")
print(f"Total data points:  {cpu_matrix.size:,}")
print(f"Average CPU (fleet):{avg_cpu.mean():.1f}%")
print(f"Servers > 80% avg:  {n_breaching}")

# 3. Broadcasting: normalize each timestamp by fleet average
fleet_avg_per_ts = cpu_matrix.mean(axis=0)  # shape: (1000,) — per-timestamp average
# Subtract fleet average from each server (broadcasting: (6000,1000) - (1000,))
deviation = cpu_matrix - fleet_avg_per_ts   # shape: (6000, 1000)

# Servers consistently ABOVE fleet average (anomalies)
consistently_high = (deviation > 10).mean(axis=1) > 0.5  # True for > 50% of timestamps
print(f"Consistently above fleet average: {consistently_high.sum()}")

In [ ]:
import numpy as np
import time

# ══════════════════════════════════════
# ANTI-PATTERN vs PATTERN
# ══════════════════════════════════════

matrix = np.random.rand(6000, 1000)

# ❌ ANTI-PATTERN: Python loop over NumPy array
start = time.perf_counter()
result_loop = []
for i in range(matrix.shape[0]):
    row_mean = 0
    for j in range(matrix.shape[1]):
        row_mean += matrix[i, j]
    result_loop.append(row_mean / matrix.shape[1])
loop_ms = (time.perf_counter() - start) * 1000

# ✅ PATTERN: Vectorized
start = time.perf_counter()
result_vec = matrix.mean(axis=1)
vec_ms = (time.perf_counter() - start) * 1000

print(f"Loop:      {loop_ms:.0f} ms")
print(f"Vectorized:{vec_ms:.2f} ms")
print(f"Speedup:   {loop_ms/vec_ms:.0f}x")
print(f"Results match: {np.allclose(result_loop, result_vec)}")

# Common vectorized patterns:
print("
Common NumPy patterns:")
arr = np.array([3, 1, 4, 1, 5, 9, 2, 6])

print(f"Cumulative sum:   {np.cumsum(arr)}")             # rolling sum
print(f"Rolling diff:     {np.diff(arr)}")               # day-over-day change
print(f"Percentile 95:    {np.percentile(arr, 95)}")     # SLA threshold
print(f"Boolean where:    {arr[arr > 4]}")               # filter values > 4
print(f"Clip to 0-5:      {np.clip(arr, 0, 5)}")        # cap values

## 🎤 5 Interview Q&A

**Q1: Why is NumPy so much faster than a Python loop?**
NumPy's array operations are implemented in C and Fortran. When you call
`arr * 2.5`, C code iterates over a contiguous block of memory — no Python
interpreter overhead, no object boxing/unboxing, no pointer chasing through
Python's heap. The speedup is typically 10-200x for arithmetic operations.

---

**Q2: What is broadcasting and when does it fail?**
Broadcasting is NumPy's rule for operating on arrays with different shapes.
Arrays are compatible if their dimensions are equal or one of them is 1 (from
the trailing dimension). It fails with `ValueError: operands could not be broadcast`
when shapes are incompatible — e.g., `(100, 3)` and `(4,)` — 3 ≠ 4 and neither is 1.
Fix by reshaping: `arr.reshape(-1, 1)` adds a trailing dimension of 1.

---

**Q3: When would you use pandas vs NumPy?**
NumPy for pure numeric array math — lowest overhead, maximum speed.
pandas for labeled data with mixed types, SQL-style group/join/pivot, time series
with DatetimeIndex. Rule of thumb: if your operation is "apply this math to every
row" and data is numeric → NumPy. If your operation involves groupby, merge, or
column names → pandas (which uses NumPy internally).

---

**Q4: What is the difference between `.copy()` and a view?**
NumPy operations often return **views** (slices, reshapes) — they share memory
with the original. Modifying a view modifies the original. `arr.copy()` creates
a new allocation. Use `.copy()` when you need to modify a subset without affecting
the source. Check with `arr.base is None` — True means it's the owner, not a view.

---

**Q5: What is vectorization in the context of pandas?**
pandas supports vectorized operations through `.apply()` (row-wise Python, slow),
`.map()` (element-wise), and direct column arithmetic (e.g., `df['a'] + df['b']`).
The fastest path is always direct column math — pandas delegates to NumPy.
`.apply()` with a Python lambda is effectively a loop — avoid for performance.

## 📚 Key Terms

| Term | Definition |
|------|------------|
| **Vectorization** | Applying an operation to an entire array at once in compiled C/Fortran code — no Python interpreter loop |
| **Broadcasting** | NumPy's rules for operating on arrays with different (but compatible) shapes without copying data |
| **dtype** | The data type of all elements in a NumPy array — `float64`, `int32`, `bool`, etc. Determines memory per element |
| **Contiguous Memory** | Elements stored in adjacent memory locations — enables SIMD CPU instructions and cache efficiency |
| **View vs Copy** | View shares memory with original; copy is independent. Slices return views; `.copy()` forces a copy |
| **axis** | The dimension along which an operation reduces. `axis=0` = across rows (per column). `axis=1` = across columns (per row) |
| **Boolean Mask** | A bool array used to select elements — `arr[arr > 5]` selects all elements > 5 |
| **np.where** | `np.where(condition, x, y)` — element-wise: x where condition is True, y where False |
| **SIMD** | Single Instruction Multiple Data — CPU feature that applies one instruction to multiple data elements simultaneously |

## 💼 The Citi Narrative

**Context:** Daily capacity analysis — 6,000 servers × 1,000 timestamps = 6M data points,
run every morning at 06:00 for the infrastructure team.

**The Problem:** Python loop implementation processed the 6M-cell matrix in ~180 seconds.
The daily batch window was 10 minutes. The capacity analysis was the bottleneck.

**The Fix:** Rewrote matrix operations using NumPy:
- Row means: `cpu_matrix.mean(axis=1)` instead of `for server in servers: sum/count`
- Threshold breaches: `(avg_cpu > 80).sum()` instead of `for x in avgs: if x > 80`
- Fleet deviation: broadcasting `cpu_matrix - fleet_avg` instead of nested loops

**Result:** 180 seconds → 1.2 seconds. 150x speedup. Zero algorithm change.
The capacity analysis went from a nightly batch to a real-time interactive query —
analysts could now ask "which servers are trending toward breach?" during standup.

**Interview Line:** *"The algorithm was correct — it was the implementation that was wrong.
Python loops over 6M NumPy elements are essentially calling the Python interpreter
6 million times. Replacing with vectorized operations cut runtime from 180 seconds to 1.2 seconds.
Same math. Different execution model."*

In [ ]:
# Practice: Rewrite these Python loops using NumPy vectorization

import numpy as np

# Setup
np.random.seed(0)
readings = np.random.uniform(20, 95, size=(500, 24))  # 500 servers, 24 hours

# ── PRACTICE 1 ──────────────────────────────────────────────────────────────
# Loop version (SLOW):
daily_avg_loop = []
for i in range(readings.shape[0]):
    total = 0
    for j in range(readings.shape[1]):
        total += readings[i, j]
    daily_avg_loop.append(total / readings.shape[1])

# Write the NumPy version here:
daily_avg_np = None  # your answer

# ── PRACTICE 2 ──────────────────────────────────────────────────────────────
# Find all servers where ANY hourly reading exceeded 90%
# Loop version:
high_alert_loop = []
for i in range(readings.shape[0]):
    for j in range(readings.shape[1]):
        if readings[i, j] > 90:
            high_alert_loop.append(i)
            break

# Write the NumPy version (one line):
high_alert_np = None  # your answer

# ── SOLUTIONS ───────────────────────────────────────────────────────────────
daily_avg_np = readings.mean(axis=1)
high_alert_np = np.where((readings > 90).any(axis=1))[0]

print(f"Practice 1 - Loop vs NumPy match: {np.allclose(daily_avg_loop, daily_avg_np)}")
print(f"Practice 2 - Servers with >90% peak: {len(high_alert_np)}")

## 🎯 Summary

### The Pattern
**Vectorization** — Replace Python loops with NumPy array operations.
Same algorithm. 10-200x faster.

### The Four Key Operations
| Operation | Python Loop | NumPy Vectorized |
|-----------|-------------|-----------------|
| Per-row mean | `for row in matrix` | `matrix.mean(axis=1)` |
| Filter rows | `for x in arr: if x > T` | `arr[arr > T]` |
| Element-wise math | `for i: arr[i] * 2.5 + 100` | `arr * 2.5 + 100` |
| Row vs baseline | `for i,j: m[i,j] - base[j]` | `matrix - baseline` (broadcast) |

### Interview Confidence Checklist
- [ ] Can explain WHY NumPy is faster (C, contiguous memory, SIMD)
- [ ] Can explain broadcasting with a concrete shape example
- [ ] Can rewrite a Python loop as a vectorized NumPy operation on the spot
- [ ] Can name the Citi narrative: 180s → 1.2s, 6M-cell capacity matrix

**"Simplicity and clarity is Gold."** — Sean's Study Mantra 🚀